In [31]:
import requests
import pandas as pd
import requests
import pandas as pd
import time
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import numpy as np
import pyogrio
import re


## Downloading csv for each state from https://www.floodsmart.gov/historical-nfip-claims-information-and-trends?map=countries/us/custom/us-all-territories&region=us&miny=all&maxy=all&county=&gtype=country and put into a folder called NFIP Insurance Claims in data/raw

In [24]:
input_dir = Path(
    "/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/"
    "data/raw/NFIP Insurance Claims"
)

csv_files = sorted(input_dir.glob("**/*.csv"))

print("TOTAL CSV FILES:", len(csv_files))

inventory = []

for file in csv_files:
    df = pd.read_csv(file)

    inventory.append({
        "State": file.parent.name.replace("NFIP_", ""),
        "File": file.name,
        "Rows": len(df),
        "Unique Counties": df["Name"].nunique(),
        "Duplicate Counties": df["Name"].duplicated().sum(),
        "Missing County Names": df["Name"].isna().sum(),
        "Missing Claim Counts": df["Total Paid Claims"].isna().sum(),
        "Missing Dollar Values": df["Total Claim Dollars Paid"].isna().sum()
    })

inventory_df = pd.DataFrame(inventory)

print("FILE INVENTORY:")
display(inventory_df)

print("FILES PER STATE:")
print(inventory_df.groupby("State").size())

print("TOTAL COUNTY RECORDS:", inventory_df["Rows"].sum())

TOTAL CSV FILES: 50
FILE INVENTORY:


,State,File,Rows,Unique Counties,Duplicate Counties,Missing County Names,Missing Claim Counts,Missing Dollar Values
0,Alabama,Alabama_1980_1989.csv,67,67,0,0,0,0
1,Alabama,Alabama_1990_1999.csv,67,67,0,0,0,0
2,Alabama,Alabama_2000_2009.csv,67,67,0,0,0,0
3,Alabama,Alabama_2010_2019.csv,67,67,0,0,0,0
4,Alabama,Alabama_2020_2025.csv,67,67,0,0,0,0
5,Arkansas,Arkansas_1980_1989.csv,75,75,0,0,0,0
6,Arkansas,Arkansas_1990_1999.csv,75,75,0,0,0,0
7,Arkansas,Arkansas_2000_2009.csv,75,75,0,0,0,0
8,Arkansas,Arkansas_2010_2019.csv,75,75,0,0,0,0
9,Arkansas,Arkansas_2020_2025.csv,75,75,0,0,0,0


FILES PER STATE:
State
Alabama          5
Arkansas         5
Iowa             5
Kentucky         5
Mississippi      5
Missouri         5
Nebraska         5
Oklahoma         5
Tennessee        5
West_Virginia    5
dtype: int64
TOTAL COUNTY RECORDS: 4390


### Add county shapes to the csvs

In [28]:
tiger_path = (
    project_dir
    / "data"
    / "raw"
    / "tiger_counties"
    / "tl_2025_us_county.shp"
)

counties = gpd.read_file(tiger_path)

print("CRS:", counties.crs)
print("Columns:", counties.columns.tolist())
print("County records:", len(counties))

display(
    counties[["GEOID", "STATEFP", "COUNTYFP", "NAME"]].head(10)
)

CRS: EPSG:4269
Columns: ['STATEFP', 'COUNTYFP', 'COUNTYNS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'LSAD', 'CLASSFP', 'MTFCC', 'CSAFP', 'CBSAFP', 'METDIVFP', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']
County records: 3235


,GEOID,STATEFP,COUNTYFP,NAME
0,40075,40,075,Kiowa
1,46079,46,079,Lake
2,37033,37,033,Caswell
3,48377,48,377,Presidio
4,39057,39,057,Greene
5,48489,48,489,Willacy
6,72021,72,021,Bayamón
7,35028,35,028,Los Alamos
8,40023,40,023,Choctaw
9,20041,20,041,Dickinson


In [29]:

nfip_dir = project_dir / "data" / "raw" / "NFIP Insurance Claims"

state_fips = {
    "Alabama": "01",
    "Arkansas": "05",
    "Iowa": "19",
    "Kentucky": "21",
    "Mississippi": "28",
    "Missouri": "29",
    "Nebraska": "31",
    "Oklahoma": "40",
    "Tennessee": "47",
    "West_Virginia": "54"
}

decades = [
    "1980_1989",
    "1990_1999",
    "2000_2009",
    "2010_2019",
    "2020_2025"
]

def normalize_county_name(name):
    return re.sub(r"\s+", " ", str(name).strip()).casefold()

tiger = counties.copy()
tiger["STATEFP"] = tiger["STATEFP"].astype(str).str.zfill(2)
tiger["county_key"] = tiger["NAME"].apply(normalize_county_name)

validation_results = []

for state, fips in state_fips.items():

    state_counties = tiger[tiger["STATEFP"] == fips].copy()

    for decade in decades:

        csv_path = (
            nfip_dir
            / f"NFIP_{state}"
            / f"{state}_{decade}.csv"
        )

        df = pd.read_csv(csv_path)
        df["county_key"] = df["Name"].apply(normalize_county_name)

        unmatched_csv = sorted(
            set(df["county_key"]) - set(state_counties["county_key"])
        )

        unmatched_tiger = sorted(
            set(state_counties["county_key"]) - set(df["county_key"])
        )

        validation_results.append({
            "State": state,
            "Decade": decade,
            "CSV Counties": len(df),
            "TIGER Counties": len(state_counties),
            "Unmatched CSV Names": len(unmatched_csv),
            "Unmatched TIGER Names": len(unmatched_tiger),
            "CSV Name Examples": unmatched_csv[:5],
            "TIGER Name Examples": unmatched_tiger[:5]
        })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("Total files checked:", len(validation_df))
print(
    "Files with exact county-name matches:",
    (
        (validation_df["Unmatched CSV Names"] == 0)
        & (validation_df["Unmatched TIGER Names"] == 0)
    ).sum()
)

,State,Decade,CSV Counties,TIGER Counties,Unmatched CSV Names,Unmatched TIGER Names,CSV Name Examples,TIGER Name Examples
0,Alabama,1980_1989,67,67,0,0,[],[]
1,Alabama,1990_1999,67,67,0,0,[],[]
2,Alabama,2000_2009,67,67,0,0,[],[]
3,Alabama,2010_2019,67,67,0,0,[],[]
4,Alabama,2020_2025,67,67,0,0,[],[]
5,Arkansas,1980_1989,75,75,0,0,[],[]
6,Arkansas,1990_1999,75,75,0,0,[],[]
7,Arkansas,2000_2009,75,75,0,0,[],[]
8,Arkansas,2010_2019,75,75,0,0,[],[]
9,Arkansas,2020_2025,75,75,0,0,[],[]


Total files checked: 50
Files with exact county-name matches: 45


In [30]:
mo_counties = counties[
    counties["STATEFP"].astype(str).str.zfill(2) == "29"
].copy()

display(
    mo_counties[
        ["GEOID", "COUNTYFP", "NAME", "NAMELSAD"]
    ].sort_values("NAME").tail(10)
)

print(
    mo_counties[
        mo_counties["NAME"].str.contains("Louis", case=False, na=False)
    ][["GEOID", "COUNTYFP", "NAME", "NAMELSAD"]]
)

,GEOID,COUNTYFP,NAME,NAMELSAD
1980,29211,211,Sullivan,Sullivan County
1384,29213,213,Taney,Taney County
3040,29215,215,Texas,Texas County
2884,29217,217,Vernon,Vernon County
2151,29219,219,Warren,Warren County
2617,29221,221,Washington,Washington County
2256,29223,223,Wayne,Wayne County
2407,29225,225,Webster,Webster County
556,29227,227,Worth,Worth County
1177,29229,229,Wright,Wright County


      GEOID COUNTYFP       NAME          NAMELSAD
1610  29510      510  St. Louis    St. Louis city
3125  29189      189  St. Louis  St. Louis County


In [32]:

output_dir = (
    project_dir
    / "data"
    / "processed"
    / "NFIP_Floods_By_State"
)

output_dir.mkdir(parents=True, exist_ok=True)

nfip_dir = project_dir / "data" / "raw" / "NFIP Insurance Claims"


state_fips = {
    "Alabama": "01",
    "Arkansas": "05",
    "Iowa": "19",
    "Kentucky": "21",
    "Mississippi": "28",
    "Missouri": "29",
    "Nebraska": "31",
    "Oklahoma": "40",
    "Tennessee": "47",
    "West_Virginia": "54"
}

state_epsg = {
    "01": 26916,
    "05": 26915,
    "19": 26915,
    "21": 26916,
    "28": 26915,
    "29": 26915,
    "31": 26914,
    "40": 26914,
    "47": 26916,
    "54": 26917
}

decades = [
    "1980_1989",
    "1990_1999",
    "2000_2009",
    "2010_2019",
    "2020_2025"
]


tiger = counties.copy()

tiger["STATEFP"] = tiger["STATEFP"].astype(str).str.zfill(2)
tiger["GEOID"] = tiger["GEOID"].astype(str).str.zfill(5)

tiger["county_key"] = (
    tiger["NAME"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

tiger.loc[
    tiger["GEOID"] == "29510",
    "county_key"
] = "st. louis city"


existing = [
    output_dir / f"{state.replace('_', '_')}_Floods.gdb"
    for state in state_fips
]

existing = [path for path in existing if path.exists()]

if existing:
    raise FileExistsError(
        "These output geodatabases already exist. "
        "Nothing was written:\n"
        + "\n".join(str(path) for path in existing)
    )



export_summary = []

for state, fips in state_fips.items():

    print("\n" + "=" * 60)
    print("STATE:", state)
    print("=" * 60)

    state_counties = tiger[
        tiger["STATEFP"] == fips
    ].copy()

    state_gdb = (
        output_dir
        / f"{state}_Floods.gdb"
    )

    state_gdb_created = False

    for decade in decades:

        print("Processing:", decade)

        csv_path = (
            nfip_dir
            / f"NFIP_{state}"
            / f"{state}_{decade}.csv"
        )

        if not csv_path.exists():
            raise FileNotFoundError(csv_path)

        df = pd.read_csv(csv_path)

        df["county_key"] = (
            df["Name"]
            .astype(str)
            .str.strip()
            .str.casefold()
        )

        if state == "Missouri":
            df.loc[
                df["county_key"] == "st. louis city",
                "county_key"
            ] = "st. louis city"

        df["Total Claim Dollars Paid"] = pd.to_numeric(
            df["Total Claim Dollars Paid"]
            .astype(str)
            .str.replace(r"[$,]", "", regex=True)
            .str.strip(),
            errors="raise"
        )

        df["Total Paid Claims"] = pd.to_numeric(
            df["Total Paid Claims"],
            errors="raise"
        ).astype("int64")

        if df["county_key"].duplicated().any():
            raise ValueError(
                f"Duplicate county names in {csv_path}"
            )

        unmatched = (
            set(df["county_key"])
            - set(state_counties["county_key"])
        )

        if unmatched:
            raise ValueError(
                f"Unmatched counties in {state}, {decade}: "
                f"{sorted(unmatched)}"
            )

        missing = (
            set(state_counties["county_key"])
            - set(df["county_key"])
        )

        if missing:
            raise ValueError(
                f"Missing TIGER counties in {state}, {decade}: "
                f"{sorted(missing)}"
            )

        layer = state_counties.merge(
            df[
                [
                    "county_key",
                    "Total Paid Claims",
                    "Total Claim Dollars Paid"
                ]
            ],
            on="county_key",
            how="left",
            validate="one_to_one"
        )

        layer = layer.drop(columns=["county_key"])

        layer["State"] = state.replace("_", " ")
        layer["Decade"] = decade

        layer = layer.to_crs(
            epsg=state_epsg[fips]
        )

        layer_name = f"NFIP_{decade}"

        pyogrio.write_dataframe(
            layer,
            state_gdb,
            layer=layer_name,
            driver="OpenFileGDB",
            geometry_type="MultiPolygon",
            promote_to_multi=True
        )

        state_gdb_created = True

        print(
            f"  Exported {layer_name}: "
            f"{len(layer)} counties | "
            f"EPSG:{state_epsg[fips]}"
        )

        export_summary.append({
            "State": state,
            "Decade": decade,
            "Counties": len(layer),
            "Claims": int(layer["Total Paid Claims"].sum()),
            "Dollars Paid": layer[
                "Total Claim Dollars Paid"
            ].sum(),
            "CRS": f"EPSG:{state_epsg[fips]}",
            "Geodatabase": state_gdb.name,
            "Layer": layer_name
        })




summary_df = pd.DataFrame(export_summary)

print("\n" + "=" * 60)
print("NFIP EXPORT COMPLETE")
print("=" * 60)

print("Geodatabases:", summary_df["Geodatabase"].nunique())
print("Layers:", len(summary_df))
print("Total county-decade records:", summary_df["Counties"].sum())

display(summary_df)


STATE: Alabama

Processing: 1980_1989


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Total Paid Claims of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Total Paid Claims' to 'Total_Paid_Claims'
  ogr_write(
/opt/anaconda3/envs/geosp

  Exported NFIP_1980_1989: 67 counties | EPSG:26916

Processing: 1990_1999
  Exported NFIP_1990_1999: 67 counties | EPSG:26916

Processing: 2000_2009
  Exported NFIP_2000_2009: 67 counties | EPSG:26916

Processing: 2010_2019
  Exported NFIP_2010_2019: 67 counties | EPSG:26916

Processing: 2020_2025


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Total Paid Claims of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Total Paid Claims' to 'Total_Paid_Claims'
  ogr_write(
/opt/anaconda3/envs/geosp

  Exported NFIP_2020_2025: 67 counties | EPSG:26916

STATE: Arkansas

Processing: 1980_1989
  Exported NFIP_1980_1989: 75 counties | EPSG:26915

Processing: 1990_1999
  Exported NFIP_1990_1999: 75 counties | EPSG:26915

Processing: 2000_2009


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Total Paid Claims of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Total Paid Claims' to 'Total_Paid_Claims'
  ogr_write(
/opt/anaconda3/envs/geosp

... (15 outputs left out)

/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Total Paid Claims of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Total Paid Claims' to 'Total_Paid_Claims'
  ogr_write(
/opt/anaconda3/envs/geosp

  Exported NFIP_1990_1999: 95 counties | EPSG:26916

Processing: 2000_2009
  Exported NFIP_2000_2009: 95 counties | EPSG:26916

Processing: 2010_2019
  Exported NFIP_2010_2019: 95 counties | EPSG:26916

Processing: 2020_2025
  Exported NFIP_2020_2025: 95 counties | EPSG:26916

STATE: West_Virginia

Processing: 1980_1989
  Exported NFIP_1980_1989: 55 counties | EPSG:26917

Processing: 1990_1999
  Exported NFIP_1990_1999: 55 counties | EPSG:26917

Processing: 2000_2009
  Exported NFIP_2000_2009: 55 counties | EPSG:26917

Processing: 2010_2019
  Exported NFIP_2010_2019: 55 counties | EPSG:26917

Processing: 2020_2025
  Exported NFIP_2020_2025: 55 counties | EPSG:26917

NFIP EXPORT COMPLETE
Geodatabases: 10
Layers: 50
Total county-decade records: 4390


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Total Paid Claims of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Total Paid Claims' to 'Total_Paid_Claims'
  ogr_write(
/opt/anaconda3/envs/geosp

,State,Decade,Counties,Claims,Dollars Paid,CRS,Geodatabase,Layer
0,Alabama,1980_1989,67,3135,3.330653e+07,EPSG:26916,Alabama_Floods.gdb,NFIP_1980_1989
1,Alabama,1990_1999,67,7732,1.472868e+08,EPSG:26916,Alabama_Floods.gdb,NFIP_1990_1999
2,Alabama,2000_2009,67,14562,7.099460e+08,EPSG:26916,Alabama_Floods.gdb,NFIP_2000_2009
3,Alabama,2010_2019,67,2799,9.379562e+07,EPSG:26916,Alabama_Floods.gdb,NFIP_2010_2019
4,Alabama,2020_2025,67,3642,1.659759e+08,EPSG:26916,Alabama_Floods.gdb,NFIP_2020_2025
5,Arkansas,1980_1989,75,1418,1.090497e+07,EPSG:26915,Arkansas_Floods.gdb,NFIP_1980_1989
6,Arkansas,1990_1999,75,915,1.055243e+07,EPSG:26915,Arkansas_Floods.gdb,NFIP_1990_1999
7,Arkansas,2000_2009,75,1747,4.215001e+07,EPSG:26915,Arkansas_Floods.gdb,NFIP_2000_2009
8,Arkansas,2010_2019,75,2933,1.178005e+08,EPSG:26915,Arkansas_Floods.gdb,NFIP_2010_2019
9,Arkansas,2020_2025,75,656,3.252546e+07,EPSG:26915,Arkansas_Floods.gdb,NFIP_2020_2025


### ^^ Now exported gdbs for each state as /processed/NFIP_Floods_By_State

#### Removing extra Fields amended from the tiger state counties shapefile

In [36]:
source_dir = (
    project_dir
    / "data"
    / "processed"
    / "NFIP_Floods_By_State"
)

clean_dir = (
    project_dir
    / "data"
    / "processed"
    / "NFIP_Floods_By_State_Clean"
)

clean_dir.mkdir(parents=True, exist_ok=True)

field_mapping = {
    "GEOID": "GEOID",
    "STATEFP": "STATEFP",
    "COUNTYFP": "COUNTYFP",
    "NAME": "NAME",
    "Total_Paid_Claims": "Total_Paid_Claims",
    "Total_Claim_Dollars_Paid": "Total_Claim_Dollars_Paid",
    "State": "State",
    "Decade": "Decade"
}

gdb_files = sorted(source_dir.glob("*.gdb"))

if not gdb_files:
    raise FileNotFoundError("No source NFIP geodatabases found.")

existing_outputs = [
    clean_dir / gdb.name
    for gdb in gdb_files
    if (clean_dir / gdb.name).exists()
]

if existing_outputs:
    raise FileExistsError(
        "Output geodatabases already exist. Nothing overwritten:\n"
        + "\n".join(str(p) for p in existing_outputs)
    )


clean_summary = []

for source_gdb in gdb_files:

    output_gdb = clean_dir / source_gdb.name

    print(f"Processing {source_gdb.name}")

    layers = pyogrio.list_layers(source_gdb)

    for layer_info in layers:

        layer_name = layer_info[0]

        gdf = pyogrio.read_dataframe(
            source_gdb,
            layer=layer_name
        )

        missing = [
            field for field in field_mapping
            if field not in gdf.columns
        ]

        if missing:
            raise ValueError(
                f"{layer_name} is missing: {missing}"
            )

        clean_gdf = gdf[
            list(field_mapping.keys()) + ["geometry"]
        ].copy()

        clean_gdf = clean_gdf.rename(
            columns=field_mapping
        )

        clean_gdf["GEOID"] = (
            clean_gdf["GEOID"].astype(str).str.zfill(5)
        )

        clean_gdf["STATEFP"] = (
            clean_gdf["STATEFP"].astype(str).str.zfill(2)
        )

        clean_gdf["COUNTYFP"] = (
            clean_gdf["COUNTYFP"].astype(str).str.zfill(3)
        )

        pyogrio.write_dataframe(
            clean_gdf,
            output_gdb,
            layer=layer_name,
            driver="OpenFileGDB",
            geometry_type="MultiPolygon",
            promote_to_multi=True
        )

        clean_summary.append({
            "Geodatabase": output_gdb.name,
            "Layer": layer_name,
            "Records": len(clean_gdf),
            "Attribute Fields": len(field_mapping),
            "CRS": str(clean_gdf.crs)
        })

        print(f"  Exported {layer_name}: {len(clean_gdf)} records")



clean_summary_df = pd.DataFrame(clean_summary)

print("CLEAN EXPORT COMPLETE")
print("Geodatabases:", clean_summary_df["Geodatabase"].nunique())
print("Layers:", len(clean_summary_df))
print("Total records:", clean_summary_df["Records"].sum())

display(clean_summary_df)

Processing Alabama_Floods.gdb
  Exported NFIP_1980_1989: 67 records
  Exported NFIP_1990_1999: 67 records
  Exported NFIP_2000_2009: 67 records
  Exported NFIP_2010_2019: 67 records
  Exported NFIP_2020_2025: 67 records
Processing Arkansas_Floods.gdb
  Exported NFIP_1980_1989: 75 records
  Exported NFIP_1990_1999: 75 records
  Exported NFIP_2000_2009: 75 records
  Exported NFIP_2010_2019: 75 records
  Exported NFIP_2020_2025: 75 records
Processing Iowa_Floods.gdb
  Exported NFIP_1980_1989: 99 records
  Exported NFIP_1990_1999: 99 records
  Exported NFIP_2000_2009: 99 records
  Exported NFIP_2010_2019: 99 records
  Exported NFIP_2020_2025: 99 records
Processing Kentucky_Floods.gdb
  Exported NFIP_1980_1989: 120 records
  Exported NFIP_1990_1999: 120 records
  Exported NFIP_2000_2009: 120 records
  Exported NFIP_2010_2019: 120 records
  Exported NFIP_2020_2025: 120 records
Processing Mississippi_Floods.gdb
  Exported NFIP_1980_1989: 82 records
  Exported NFIP_1990_1999: 82 records
  Expo

,Geodatabase,Layer,Records,Attribute Fields,CRS
0,Alabama_Floods.gdb,NFIP_1980_1989,67,8,EPSG:26916
1,Alabama_Floods.gdb,NFIP_1990_1999,67,8,EPSG:26916
2,Alabama_Floods.gdb,NFIP_2000_2009,67,8,EPSG:26916
3,Alabama_Floods.gdb,NFIP_2010_2019,67,8,EPSG:26916
4,Alabama_Floods.gdb,NFIP_2020_2025,67,8,EPSG:26916
5,Arkansas_Floods.gdb,NFIP_1980_1989,75,8,EPSG:26915
6,Arkansas_Floods.gdb,NFIP_1990_1999,75,8,EPSG:26915
7,Arkansas_Floods.gdb,NFIP_2000_2009,75,8,EPSG:26915
8,Arkansas_Floods.gdb,NFIP_2010_2019,75,8,EPSG:26915
9,Arkansas_Floods.gdb,NFIP_2020_2025,75,8,EPSG:26915
